In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D5 — IMF Country Focus: Booming India at Risk of Overheating
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip install PyMuPDF
from google.colab import files
from pathlib import Path

import hashlib
import json
import platform
import re
import sys

import fitz

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D5"

DOCUMENT_NAME = (
    "IMF Country Focus — Booming India at risk of overheating"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

SOURCE_FORMAT = ".pdf"

INPUT_REPRESENTATION = "Original PDF document"

EXPECTED_PAGE_COUNT = 2

# ------------------------------------------------------------
# Stage 1 reference expectations
# Used only for post-extraction diagnostics / later validation.
# These values must NOT be disclosed in the model prompt.
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = 44

EXPECTED_CATEGORY_COUNTS = {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 16,
    "Statistical table observation": 18
}

EXPECTED_FIELDS = [
    "Category",
    "Indicator or Policy Area",
    "Value",
    "Unit",
    "Qualifier",
    "Reference Period",
    "Description",
    "Source Location"
]

ALLOWED_CATEGORIES = set(
    EXPECTED_CATEGORY_COUNTS.keys()
)

OUTPUT_DIR = Path(
    "outputs_D5_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Branch name:", BRANCH_NAME)
print("Input representation:", INPUT_REPRESENTATION)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 2. Source document upload
# ============================================================

print(
    "Upload the original D5 PDF."
)

uploaded = files.upload()

pdf_files = [
    Path(filename)
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if len(pdf_files) != 1:

    raise ValueError(
        "Upload exactly one PDF file."
    )

SOURCE_PATH = pdf_files[0]

print(
    "Uploaded source:",
    SOURCE_PATH.name
)

print(
    "File size:",
    f"{SOURCE_PATH.stat().st_size:,} bytes"
)

In [ ]:
# ============================================================
# 3. Source SHA-256
# ============================================================

def sha256_file(path):
    """
    Return the SHA-256 hash of a file.
    """

    digest = hashlib.sha256()

    with path.open("rb") as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

print(
    "Source SHA-256:",
    SOURCE_SHA256
)

In [ ]:
# ============================================================
# 4. Source PDF diagnostics
# ============================================================

document = fitz.open(
    SOURCE_PATH
)

page_count = len(
    document
)

page_count_valid = (
    page_count
    == EXPECTED_PAGE_COUNT
)

pdf_is_encrypted = bool(
    document.is_encrypted
)

pdf_needs_password = bool(
    document.needs_pass
)

file_non_empty = (
    SOURCE_PATH.exists()
    and SOURCE_PATH.stat().st_size > 0
)


print(
    "Page count:",
    page_count
)

print(
    "Expected page count:",
    EXPECTED_PAGE_COUNT
)

print(
    "Page count valid:",
    page_count_valid
)

print(
    "Encrypted:",
    pdf_is_encrypted
)

print(
    "Needs password:",
    pdf_needs_password
)

print(
    "File non-empty:",
    file_non_empty
)


if pdf_needs_password:

    raise ValueError(
        "The source PDF requires a password."
    )

if not page_count_valid:

    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, "
        f"found {page_count}."
    )

In [ ]:
# ============================================================
# 5. Text-layer diagnostics
# ============================================================

page_text_character_counts = []

page_word_counts = []

page_image_counts = []

all_page_text = []


for page_index, page in enumerate(
    document
):

    page_text = page.get_text(
        "text"
    )

    page_text_character_counts.append(
        len(
            page_text
        )
    )

    page_word_counts.append(
        len(
            page_text.split()
        )
    )

    page_image_counts.append(
        len(
            page.get_images(
                full=True
            )
        )
    )

    all_page_text.append(
        page_text
    )


full_text = "\n".join(
    all_page_text
)

text_layer_present = bool(
    full_text.strip()
)

ocr_required = not text_layer_present


print(
    "Text layer present:",
    text_layer_present
)

print(
    "OCR required:",
    ocr_required
)

print(
    "Text characters by page:",
    page_text_character_counts
)

print(
    "Words by page:",
    page_word_counts
)

print(
    "Embedded images by page:",
    page_image_counts
)

In [ ]:
# ============================================================
# 6. Source component verification
# ============================================================

EXPECTED_COMPONENTS = {
    "article_title":
        "Booming India at risk of overheating",

    "country_profile":
        "India at a glance",

    "chart":
        "Taking off",

    "monetary_section":
        "Ease up on the monetary accelerator",

    "fiscal_section":
        "Reduce debt to finance development",

    "capital_markets_section":
        "Develop broader and deeper capital markets",

    "employment_section":
        "Promote job growth and bolster the infrastructure",

    "statistical_table":
        "Inflation risks",

    "author":
        "Charles Kramer",

    "publication_date":
        "April 11, 2007"
}


component_checks = {
    component_name:
        component_text in full_text

    for component_name, component_text
    in EXPECTED_COMPONENTS.items()
}


all_expected_components_present = all(
    component_checks.values()
)


print(
    json.dumps(
        component_checks,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "All expected components present:",
    all_expected_components_present
)

In [ ]:
# ============================================================
# 7. Source integrity report
# ============================================================

input_integrity_passed = all(
    [
        file_non_empty,
        page_count_valid,
        not pdf_needs_password,
        text_layer_present,
        all_expected_components_present
    ]
)


INPUT_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_file":
        SOURCE_PATH.name,

    "input_file_sha256":
        SOURCE_SHA256,

    "input_representation":
        INPUT_REPRESENTATION,

    "file_size_bytes":
        int(
            SOURCE_PATH.stat().st_size
        ),

    "file_non_empty":
        bool(
            file_non_empty
        ),

    "page_count":
        page_count,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "page_count_valid":
        bool(
            page_count_valid
        ),

    "encrypted":
        pdf_is_encrypted,

    "password_required":
        pdf_needs_password,

    "text_layer_present":
        bool(
            text_layer_present
        ),

    "ocr_required":
        bool(
            ocr_required
        ),

    "page_text_character_counts":
        page_text_character_counts,

    "page_word_counts":
        page_word_counts,

    "page_embedded_image_counts":
        page_image_counts,

    "expected_component_checks":
        component_checks,

    "all_expected_components_present":
        bool(
            all_expected_components_present
        ),

    "direct_pdf_ingestion_usable":
        bool(
            input_integrity_passed
        ),

    "input_integrity_passed":
        bool(
            input_integrity_passed
        )
}


INPUT_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D5_branch_A_input_integrity.json"
)


with INPUT_INTEGRITY_PATH.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        INPUT_INTEGRITY,
        file,
        ensure_ascii=False,
        indent=2
    )


print(
    json.dumps(
        INPUT_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)


if not input_integrity_passed:

    raise AssertionError(
        "The D5 PDF failed one or more input-integrity checks."
    )

In [ ]:
# ============================================================
# 8. Branch A representation
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        SOURCE_PATH.name,

    "input_format":
        SOURCE_FORMAT,

    "diagnostic_pdf_text_inspection_applied":
        True,

    "pdf_to_text_conversion_applied":
        False,

    "derived_representation_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "page_extraction_applied":
        False,

    "layout_reconstruction_applied":
        False,

    "table_conversion_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "model_input_description": (
        "The complete original two-page PDF is submitted directly "
        "to the LLM. PyMuPDF text extraction is used only for "
        "source-integrity diagnostics and is not supplied to the "
        "model as an alternative representation."
    )
}


REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D5_branch_A_representation.json"
)


with REPRESENTATION_PATH.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        REPRESENTATION,
        file,
        ensure_ascii=False,
        indent=2
    )


print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 9. Extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id": "D5",
    "branch": "A",
    "records": [
        {
            "Category": None,
            "Indicator or Policy Area": None,
            "Value": None,
            "Unit": None,
            "Qualifier": None,
            "Reference Period": None,
            "Description": None,
            "Source Location": None
        }
    ]
}

In [ ]:
# ============================================================
# 10. Fixed extraction task
# ============================================================

EXTRACTION_TASK = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "scope": (
        "Principal policy measures, country-profile items, "
        "explicit narrative quantitative observations, "
        "the explicit Taking off chart-caption statement, "
        "and Inflation risks table observations."
    )
}

In [ ]:
# ============================================================
# 11. Extraction prompt
# ============================================================

PROMPT_TEXT = """You are an information extraction assistant.

Extract the policy and quantitative records represented within the
defined scope of the attached original PDF article, “Booming India at
risk of overheating.”

Treat the attached original PDF as the only source of information.

Include:

1. Each principal policy measure introduced by the sentence
   “A combination of these four main policy measures is critical”.
2. Every item represented in the “India at a glance” country-profile
   box.
3. Every explicitly stated quantitative observation in the narrative
   article that belongs to the defined extraction scope.
4. The explicit quantitative statement represented in the “Taking off”
   chart caption.
5. Every numeric observation represented in the “Inflation risks”
   statistical table.

Exclude:

- values inferred or estimated from plotted chart lines;
- chart axis values;
- chart series dates and chart legend dates;
- page numbers and publication metadata;
- the photograph and photograph caption;
- promotional content;
- copyright text and publisher branding;
- values appearing only in source citations;
- qualitative statements that are not one of the principal policy
  measures;
- detailed policy sub-actions that merely elaborate a principal policy
  measure.

For every included record, extract:

- Category
- Indicator or Policy Area
- Value
- Unit
- Qualifier
- Reference Period
- Description
- Source Location

Category rules:

Use exactly one of:

- Main policy measure
- Country profile
- Narrative quantitative observation
- Statistical table observation

Indicator or Policy Area:

- Provide a concise source-grounded name for the policy area,
  country-profile item or quantitative indicator.
- Do not merge separate observations merely because they concern a
  similar economic topic.

Value:

- Preserve the explicitly represented source value.
- Use a JSON number for quantitative values.
- Use a JSON string for textual values.
- Keep negative numbers negative.
- Do not calculate, convert or reinterpret values.
- Do not infer values from chart lines.

Unit:

- Preserve the source-grounded measurement unit.
- Use "text" for textual policy or profile values.
- Do not silently correct source units.
- Preserve the unit printed in the source even if it appears unusual.

Qualifier:

- Preserve an explicit qualifier when it is directly associated with
  the extracted value.
- Use null when no qualifier is explicitly associated with the record.

Reference Period:

- Preserve explicitly associated fiscal years, years, durations or
  relative periods.
- Use null when the article provides no explicit reference period.

Description:

- Provide a concise source-grounded explanation of what the record
  represents.
- Do not introduce external interpretation.

Source Location:

Use one of these exact source-location labels when applicable:

- Page 1 — Four main policy measures
- Page 1 — India at a glance
- Page 1 — Opening narrative
- Page 1 — Taking off chart caption
- Page 1 — Ease up on the monetary accelerator
- Page 1 — Reduce debt to finance development
- Page 2 — Reduce debt to finance development
- Page 2 — Promote job growth and bolster the infrastructure
- Page 2 — Inflation risks table

Additional extraction rules:

- Use both the visible document layout and textual content.
- Respect the PDF's visual reading order.
- Distinguish narrative observations from statistical-table
  observations.
- Preserve decimal precision as represented.
- Preserve negative signs.
- Do not use external knowledge.
- Do not correct values or units based on plausibility.
- Do not follow hyperlinks.
- Do not infer values that are not explicitly represented.
- Do not estimate values from chart lines.
- Return one record for every included source observation.
- Verify that all content within the defined scope has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names defined in the schema.

Expected JSON schema:

{
  "document_id": "D5",
  "branch": "A",
  "records": [
    {
      "Category": null,
      "Indicator or Policy Area": null,
      "Value": null,
      "Unit": null,
      "Qualifier": null,
      "Reference Period": null,
      "Description": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
"""


PROMPT_PATH = (
    OUTPUT_DIR
    / "D5_branch_A_prompt.txt"
)


PROMPT_PATH.write_text(
    PROMPT_TEXT,
    encoding="utf-8"
)


PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)


print(
    PROMPT_TEXT
)

print(
    "\nPrompt saved:",
    PROMPT_PATH
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

In [ ]:
# ============================================================
# 12. Experiment metadata
# ============================================================

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_structure": {
        "expected_page_count":
            EXPECTED_PAGE_COUNT,

        "observed_page_count":
            page_count,

        "page_count_verified":
            bool(
                page_count_valid
            ),

        "machine_readable_text_layer":
            bool(
                text_layer_present
            ),

        "expected_components_verified":
            bool(
                all_expected_components_present
            )
    },

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        True,

    "diagnostic_text_extraction_applied":
        True,

    "text_extraction_used_as_model_input":
        False,

    "pdf_to_text_conversion_applied":
        False,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "page_extraction_applied":
        False,

    "layout_reconstruction_applied":
        False,

    "table_conversion_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "hyphenation_cleaning_applied":
        False,

    "reading_order_reconstruction_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "source_content_modification_applied":
        False,

    "complete_original_pdf_supplied":
        True,

    # Stage 1 reference expectations retained for
    # post-extraction diagnostics only.
    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS,

        "expected_fields":
            EXPECTED_FIELDS
    },

    "reference_expectations_disclosed_to_model":
        False,

    "input_integrity_file":
        INPUT_INTEGRITY_PATH.name,

    "input_integrity_passed":
        bool(
            input_integrity_passed
        ),

    "representation_file":
        REPRESENTATION_PATH.name,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        "JSON object with document_id, branch and records",

    "execution_environment":
        "Independent ChatGPT conversation",

    "notes": (
        "Branch A submits the complete original D5 PDF directly "
        "to the model. PyMuPDF text extraction is used only for "
        "source-integrity diagnostics. No PDF-to-text conversion, "
        "OCR, structural conversion, layout reconstruction, table "
        "conversion, cleaning or normalisation is applied before "
        "model extraction. Stage 1 reference values and expected "
        "record counts are not supplied to the model."
    )
}


EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D5_branch_A_experiment_metadata.json"
)


with EXPERIMENT_METADATA_PATH.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        EXPERIMENT_METADATA,
        file,
        ensure_ascii=False,
        indent=2
    )


print(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    )
)

## Independent Branch A extraction

Open a new independent ChatGPT conversation.

Upload:

1. the complete original D5 PDF;
2. `D5_branch_A_prompt.txt`.

Paste the complete prompt once and submit it.

Save the complete, unmodified model response as:

`D5_branch_A_raw_response.txt`

In [ ]:
# ============================================================
# 13. Raw response upload
# ============================================================

print(
    "Upload the complete unmodified model response as TXT."
)

uploaded_output = files.upload()


txt_files = [
    Path(filename)

    for filename in uploaded_output.keys()

    if filename.lower().endswith(
        ".txt"
    )
]


if len(txt_files) != 1:

    raise ValueError(
        "Upload exactly one TXT raw-response file."
    )


RAW_RESPONSE_SOURCE_PATH = txt_files[0]


print(
    "Uploaded raw response:",
    RAW_RESPONSE_SOURCE_PATH.name
)

In [ ]:
# ============================================================
# 14. Raw-response preservation
# ============================================================

RAW_RESPONSE_TEXT = (
    RAW_RESPONSE_SOURCE_PATH.read_text(
        encoding="utf-8"
    )
)


if not RAW_RESPONSE_TEXT.strip():

    raise ValueError(
        "The uploaded raw model response is empty."
    )


RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D5_branch_A_raw_response.txt"
)


RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


print(
    "Raw response preserved:",
    RAW_RESPONSE_PATH
)

print(
    "Raw-response SHA-256:",
    RAW_RESPONSE_SHA256
)

In [ ]:
# ============================================================
# 15. Raw-response parsing
# ============================================================

valid_json = False

json_parsing_error = None

PARSED_EXTRACTION = None


try:

    PARSED_EXTRACTION = json.loads(
        RAW_RESPONSE_TEXT
    )

    valid_json = True


except json.JSONDecodeError as error:

    json_parsing_error = str(
        error
    )


print(
    "Valid JSON:",
    valid_json
)

print(
    "JSON parsing error:",
    json_parsing_error
)

In [ ]:
# ============================================================
# 16. Top-level structure diagnostics
# ============================================================

top_level_object_valid = False

document_id_present = False
document_id_correct = False

branch_present = False
branch_correct = False

records_present = False
records_is_list = False

records_evaluable = False

extracted_records = []


if (
    valid_json
    and isinstance(
        PARSED_EXTRACTION,
        dict
    )
):

    top_level_object_valid = True

    document_id_present = (
        "document_id"
        in PARSED_EXTRACTION
    )

    document_id_correct = (
        PARSED_EXTRACTION.get(
            "document_id"
        )
        == DOCUMENT_ID
    )

    branch_present = (
        "branch"
        in PARSED_EXTRACTION
    )

    branch_correct = (
        PARSED_EXTRACTION.get(
            "branch"
        )
        == BRANCH
    )

    records_present = (
        "records"
        in PARSED_EXTRACTION
    )

    records_is_list = isinstance(
        PARSED_EXTRACTION.get(
            "records"
        ),
        list
    )

    if records_is_list:

        extracted_records = (
            PARSED_EXTRACTION[
                "records"
            ]
        )


records_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list
])


number_of_records = len(
    extracted_records
)


record_count_valid = (
    number_of_records
    == EXPECTED_RECORD_COUNT

    if records_evaluable

    else None
)


print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Document ID present:",
    document_id_present
)

print(
    "Document ID correct:",
    document_id_correct
)

print(
    "Branch present:",
    branch_present
)

print(
    "Branch correct:",
    branch_correct
)

print(
    "Records present:",
    records_present
)

print(
    "Records is list:",
    records_is_list
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed records:",
    (
        number_of_records

        if records_evaluable

        else None
    )
)

print(
    "Expected records:",
    EXPECTED_RECORD_COUNT
)

print(
    "Record count valid:",
    record_count_valid
)

In [ ]:
# ============================================================
# 17. Parsed extraction
# ============================================================

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D5_branch_A_parsed_extraction.json"
)


if records_evaluable:

    with PARSED_EXTRACTION_PATH.open(
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            PARSED_EXTRACTION,
            file,
            ensure_ascii=False,
            indent=2
        )


    PARSED_EXTRACTION_SHA256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )


    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH
    )

    print(
        "Parsed extraction SHA-256:",
        PARSED_EXTRACTION_SHA256
    )


else:

    PARSED_EXTRACTION_SHA256 = None

    print(
        "No parsed extraction created because "
        "the response does not contain an "
        "evaluable records structure."
    )

In [ ]:
# ============================================================
# 18. Record-structure diagnostics
# ============================================================

record_structure_issues = []


for record_index, record in enumerate(
    extracted_records
):

    issues = []

    if not isinstance(
        record,
        dict
    ):

        issues.append(
            "Record is not a JSON object."
        )

    else:

        actual_fields = set(
            record.keys()
        )

        expected_fields = set(
            EXPECTED_FIELDS
        )

        missing_fields = sorted(
            expected_fields
            - actual_fields
        )

        extra_fields = sorted(
            actual_fields
            - expected_fields
        )

        if missing_fields:

            issues.append(
                {
                    "missing_fields":
                        missing_fields
                }
            )

        if extra_fields:

            issues.append(
                {
                    "extra_fields":
                        extra_fields
                }
            )

    if issues:

        record_structure_issues.append(
            {
                "record_index":
                    record_index,

                "issues":
                    issues
            }
        )


records_with_structure_issues = len(
    record_structure_issues
)


print(
    "Records with structure issues:",
    records_with_structure_issues
)


if record_structure_issues:

    print(
        json.dumps(
            record_structure_issues[:10],
            ensure_ascii=False,
            indent=2
        )
    )

In [ ]:
# ============================================================
# 19. Field-type diagnostics
# ============================================================

field_type_issues = []


for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(
        record,
        dict
    ):

        continue

    for field in [
        "Category",
        "Indicator or Policy Area",
        "Unit",
        "Qualifier",
        "Reference Period",
        "Description",
        "Source Location"
    ]:

        value = record.get(
            field
        )

        if (
            value is not None
            and not isinstance(
                value,
                str
            )
        ):

            field_type_issues.append(
                {
                    "record_index":
                        record_index,

                    "field":
                        field,

                    "observed_type":
                        type(
                            value
                        ).__name__,

                    "expected_type":
                        "string or null"
                }
            )

    value = record.get(
        "Value"
    )

    if (
        isinstance(
            value,
            bool
        )
        or (
            value is not None
            and not isinstance(
                value,
                (
                    str,
                    int,
                    float
                )
            )
        )
    ):

        field_type_issues.append(
            {
                "record_index":
                    record_index,

                "field":
                    "Value",

                "observed_type":
                    type(
                        value
                    ).__name__,

                "expected_type":
                    "string, number or null"
            }
        )


records_with_type_issues = len(
    {
        issue[
            "record_index"
        ]

        for issue in field_type_issues
    }
)


print(
    "Records with type issues:",
    records_with_type_issues
)

print(
    "Field type issue count:",
    len(
        field_type_issues
    )
)


if field_type_issues:

    print(
        json.dumps(
            field_type_issues[:20],
            ensure_ascii=False,
            indent=2
        )
    )

In [ ]:
# ============================================================
# 20. Category/count diagnostics
# ============================================================

category_issues = []


for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(
        record,
        dict
    ):

        continue

    category = record.get(
        "Category"
    )

    if category not in ALLOWED_CATEGORIES:

        category_issues.append(
            {
                "record_index":
                    record_index,

                "observed_category":
                    category
            }
        )


if records_evaluable:

    observed_category_counts = {
        category: sum(
            1

            for record in extracted_records

            if (
                isinstance(
                    record,
                    dict
                )
                and record.get(
                    "Category"
                )
                == category
            )
        )

        for category
        in EXPECTED_CATEGORY_COUNTS
    }


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


else:

    observed_category_counts = None

    category_counts_valid = None


print(
    "Category issues:",
    (
        len(
            category_issues
        )

        if records_evaluable

        else None
    )
)

print(
    "Expected category counts:"
)

print(
    json.dumps(
        EXPECTED_CATEGORY_COUNTS,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "Observed category counts:"
)

print(
    json.dumps(
        observed_category_counts,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "Category counts valid:",
    category_counts_valid
)

In [ ]:
# ============================================================
# 21. Missing-value diagnostics
# ============================================================

missing_values_by_field = {
    field: sum(
        1

        for record in extracted_records

        if (
            not isinstance(
                record,
                dict
            )
            or record.get(
                field
            ) is None
        )
    )

    for field in EXPECTED_FIELDS
}


mandatory_fields = [
    "Category",
    "Indicator or Policy Area",
    "Value",
    "Unit",
    "Description",
    "Source Location"
]


missing_mandatory_value_count = sum(
    missing_values_by_field[
        field
    ]

    for field in mandatory_fields
)


print(
    "Missing values by field:"
)

print(
    json.dumps(
        missing_values_by_field,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "Missing mandatory values:",
    missing_mandatory_value_count
)

In [ ]:
# ============================================================
# 22. Qualifier diagnostics
# ============================================================

ALLOWED_QUALIFIERS = {
    "about",
    "around",
    "just over",
    "more than",
    "nearly",
    None
}


qualifier_issues = []


for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(
        record,
        dict
    ):

        continue

    qualifier = record.get(
        "Qualifier"
    )

    if qualifier not in ALLOWED_QUALIFIERS:

        qualifier_issues.append(
            {
                "record_index":
                    record_index,

                "indicator":
                    record.get(
                        "Indicator or Policy Area"
                    ),

                "observed_qualifier":
                    qualifier
            }
        )


print(
    "Unexpected qualifier count:",
    len(
        qualifier_issues
    )
)


if qualifier_issues:

    print(
        json.dumps(
            qualifier_issues,
            ensure_ascii=False,
            indent=2
        )
    )

In [ ]:
# ============================================================
# 23. Source-location diagnostics
# ============================================================

ALLOWED_SOURCE_LOCATIONS = {
    "Page 1 — Four main policy measures",
    "Page 1 — India at a glance",
    "Page 1 — Opening narrative",
    "Page 1 — Taking off chart caption",
    "Page 1 — Ease up on the monetary accelerator",
    "Page 1 — Reduce debt to finance development",
    "Page 2 — Reduce debt to finance development",
    "Page 2 — Promote job growth and bolster the infrastructure",
    "Page 2 — Inflation risks table"
}


source_location_issues = []


for record_index, record in enumerate(
    extracted_records
):

    if not isinstance(
        record,
        dict
    ):

        continue

    source_location = record.get(
        "Source Location"
    )

    if source_location not in (
        ALLOWED_SOURCE_LOCATIONS
    ):

        source_location_issues.append(
            {
                "record_index":
                    record_index,

                "indicator":
                    record.get(
                        "Indicator or Policy Area"
                    ),

                "observed_source_location":
                    source_location
            }
        )


source_locations_valid = (
    len(
        source_location_issues
    )
    == 0
)


print(
    "Source locations valid:",
    source_locations_valid
)

print(
    "Source-location issues:",
    len(
        source_location_issues
    )
)

In [ ]:
# ============================================================
# 24. Duplicate-record diagnostics
# ============================================================

def create_record_key(record):
    """
    Create a strict structural record key.
    """

    if not isinstance(
        record,
        dict
    ):

        return None

    return (
        record.get(
            "Category"
        ),
        record.get(
            "Indicator or Policy Area"
        ),
        record.get(
            "Reference Period"
        ),
        record.get(
            "Source Location"
        )
    )


record_keys = [
    create_record_key(
        record
    )
    for record in extracted_records
]


record_key_counts = {}

for key in record_keys:

    if key is None:

        continue

    record_key_counts[key] = (
        record_key_counts.get(
            key,
            0
        )
        + 1
    )


duplicate_record_keys = sorted(
    [
        key
        for key, count
        in record_key_counts.items()
        if count > 1
    ],
    key=lambda value: str(
        value
    )
)


print(
    "Duplicate record-key count:",
    len(
        duplicate_record_keys
    )
)

In [ ]:
# ============================================================
# 25. Statistical-table diagnostics
# ============================================================

table_records = [
    record

    for record in extracted_records

    if (
        isinstance(record, dict)
        and record.get("Category")
        == "Statistical table observation"
    )
]


EXPECTED_BASE_TABLE_PERIODS = {
    "2004/05",
    "2005/06",
    "2006/07"
}


def extract_base_fiscal_period(value):
    """
    Extract the leading fiscal period from a Reference Period.

    Examples:
    - '2006/07' -> '2006/07'
    - '2006/07; as of week ended March 24, 2007'
      -> '2006/07'

    This is used only for structural validation.
    The original extracted value remains unchanged and will
    be evaluated later in Validation A.
    """

    if value is None:
        return None

    match = re.search(
        r"\b(?:19|20)\d{2}/\d{2}\b",
        str(value)
    )

    if match:
        return match.group(0)

    return None


# ------------------------------------------------------------
# Collect original indicators and base fiscal periods
# ------------------------------------------------------------

observed_table_indicators = {
    record.get("Indicator or Policy Area")

    for record in table_records

    if record.get("Indicator or Policy Area") is not None
}


observed_original_periods = {
    record.get("Reference Period")

    for record in table_records

    if record.get("Reference Period") is not None
}


observed_base_periods = {
    extract_base_fiscal_period(
        record.get("Reference Period")
    )

    for record in table_records
}


observed_base_periods.discard(None)


# ------------------------------------------------------------
# Count records by indicator and base fiscal period
# ------------------------------------------------------------

table_indicator_counts = {}

table_base_period_counts = {}


for record in table_records:

    indicator = record.get(
        "Indicator or Policy Area"
    )

    base_period = extract_base_fiscal_period(
        record.get("Reference Period")
    )

    table_indicator_counts[indicator] = (
        table_indicator_counts.get(
            indicator,
            0
        )
        + 1
    )

    table_base_period_counts[base_period] = (
        table_base_period_counts.get(
            base_period,
            0
        )
        + 1
    )


# ------------------------------------------------------------
# Detect duplicate indicator/base-period combinations
# ------------------------------------------------------------

table_indicator_period_pairs = [
    (
        record.get("Indicator or Policy Area"),
        extract_base_fiscal_period(
            record.get("Reference Period")
        )
    )

    for record in table_records
]


duplicate_table_pairs = sorted(
    {
        pair

        for pair in table_indicator_period_pairs

        if table_indicator_period_pairs.count(
            pair
        ) > 1
    },
    key=lambda value: str(value)
)


# ------------------------------------------------------------
# Identify records containing additional period detail
# ------------------------------------------------------------

period_format_differences = []


for record_index, record in enumerate(
    table_records
):

    original_period = record.get(
        "Reference Period"
    )

    base_period = extract_base_fiscal_period(
        original_period
    )

    if (
        original_period is not None
        and base_period is not None
        and str(original_period).strip()
        != base_period
    ):

        period_format_differences.append(
            {
                "table_record_index":
                    record_index,

                "indicator":
                    record.get(
                        "Indicator or Policy Area"
                    ),

                "original_reference_period":
                    original_period,

                "base_fiscal_period":
                    base_period
            }
        )


# ------------------------------------------------------------
# Structural expectations
# ------------------------------------------------------------

table_record_count_valid = (
    len(table_records)
    == 18
)

table_indicator_count_valid = (
    len(observed_table_indicators)
    == 6
)

table_base_periods_valid = (
    observed_base_periods
    == EXPECTED_BASE_TABLE_PERIODS
)

each_indicator_has_three_periods = all(
    count == 3

    for count in table_indicator_counts.values()
)

each_period_has_six_indicators = all(
    table_base_period_counts.get(
        period,
        0
    )
    == 6

    for period in EXPECTED_BASE_TABLE_PERIODS
)

table_pairs_unique = (
    len(duplicate_table_pairs)
    == 0
)


table_structurally_evaluable = all(
    [
        table_record_count_valid,
        table_indicator_count_valid,
        table_base_periods_valid,
        each_indicator_has_three_periods,
        each_period_has_six_indicators,
        table_pairs_unique
    ]
)


print(
    "Table records:",
    len(table_records)
)

print(
    "Table record count valid:",
    table_record_count_valid
)

print(
    "Observed table indicators:",
    sorted(
        observed_table_indicators,
        key=lambda value: str(value)
    )
)

print(
    "Number of unique indicators:",
    len(observed_table_indicators)
)

print(
    "Indicator count valid:",
    table_indicator_count_valid
)

print(
    "Original extracted periods:",
    sorted(
        observed_original_periods,
        key=lambda value: str(value)
    )
)

print(
    "Base fiscal periods:",
    sorted(
        observed_base_periods,
        key=lambda value: str(value)
    )
)

print(
    "Base fiscal periods valid:",
    table_base_periods_valid
)

print(
    "Indicator counts:"
)

print(
    json.dumps(
        table_indicator_counts,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "Base-period counts:"
)

print(
    json.dumps(
        table_base_period_counts,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "Each indicator has three periods:",
    each_indicator_has_three_periods
)

print(
    "Each period has six indicators:",
    each_period_has_six_indicators
)

print(
    "Duplicate indicator-period pairs:",
    len(duplicate_table_pairs)
)

print(
    "Period-format differences:",
    len(period_format_differences)
)

if period_format_differences:

    print(
        json.dumps(
            period_format_differences,
            ensure_ascii=False,
            indent=2
        )
    )

print(
    "Table Structurally evaluable:",
    table_structurally_evaluable
)

In [ ]:
# ============================================================
# 26. Branch-specific content diagnostics
# ============================================================

separate_daily_threshold_records = [
    record

    for record in extracted_records

    if (
        isinstance(
            record,
            dict
        )
        and str(
            record.get(
                "Indicator or Policy Area",
                ""
            )
        ).strip().casefold()
        == "daily income threshold"
    )
]


gross_reserve_records = [
    record

    for record in extracted_records

    if (
        isinstance(
            record,
            dict
        )
        and record.get(
            "Category"
        )
        == "Statistical table observation"
        and record.get(
            "Indicator or Policy Area"
        )
        == "Gross reserves"
    )
]


gross_reserve_units = sorted(
    {
        record.get(
            "Unit"
        )
        for record in gross_reserve_records
    },
    key=lambda value: str(
        value
    )
)


gross_reserve_unit_valid = all(
    record.get(
        "Unit"
    )
    == "million dollars"

    for record in gross_reserve_records
) and len(
    gross_reserve_records
) == 3


daily_threshold_separate_record_detected = (
    len(
        separate_daily_threshold_records
    )
    > 0
)


print(
    "Separate daily-threshold record detected:",
    daily_threshold_separate_record_detected
)

print(
    "Gross reserve records:",
    len(
        gross_reserve_records
    )
)

print(
    "Gross reserve units:",
    gross_reserve_units
)

print(
    "Gross reserve unit valid:",
    gross_reserve_unit_valid
)

In [ ]:
# ============================================================
# 27. Unsupported-chart diagnostics
# ============================================================

allowed_chart_caption_indicator = (
    "GDP growth since 2002"
)


chart_source_records = [
    record

    for record in extracted_records

    if (
        isinstance(
            record,
            dict
        )
        and record.get(
            "Source Location"
        )
        == "Page 1 — Taking off chart caption"
    )
]


unsupported_chart_records = [
    record

    for record in chart_source_records

    if record.get(
        "Indicator or Policy Area"
    )
    != allowed_chart_caption_indicator
]


chart_caption_record_count_valid = (
    len(
        chart_source_records
    )
    == 1
)

unsupported_chart_estimation_detected = (
    len(
        unsupported_chart_records
    )
    > 0
)


print(
    "Chart-caption records:",
    len(
        chart_source_records
    )
)

print(
    "Chart-caption record count valid:",
    chart_caption_record_count_valid
)

print(
    "Unsupported chart records:",
    len(
        unsupported_chart_records
    )
)

In [ ]:
# ============================================================
# 28. Structural evaluability
# ============================================================

STRUCTURAL_CHECKS = {
    "valid_json":
        bool(
            valid_json
        ),

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_present":
        bool(
            document_id_present
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_present":
        bool(
            branch_present
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_present":
        bool(
            records_present
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "record_schema_valid":
        (
            records_with_structure_issues
            == 0

            if records_evaluable

            else None
        ),

    "field_types_valid":
        (
            records_with_type_issues
            == 0

            if records_evaluable

            else None
        )
}


structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    (
        records_with_structure_issues
        == 0
    )
    if records_evaluable
    else False,
    (
        records_with_type_issues
        == 0
    )
    if records_evaluable
    else False
])


# ------------------------------------------------------------
# Reference / content diagnostics
#
# These evaluate characteristics of the extracted content.
# ------------------------------------------------------------

CONTENT_DIAGNOSTICS = {
    "record_count_matches_reference":
        (
            record_count_valid
            if records_evaluable
            else None
        ),

    "categories_valid":
        (
            len(
                category_issues
            )
            == 0
            if records_evaluable
            else None
        ),

    "category_counts_match_reference":
        (
            category_counts_valid
            if records_evaluable
            else None
        ),

    "table_structure_matches_reference":
        (
            table_structurally_evaluable
            if records_evaluable
            else None
        ),

    "missing_mandatory_value_count":
        (
            missing_mandatory_value_count
            if records_evaluable
            else None
        ),

    "unexpected_qualifier_count":
        (
            len(
                qualifier_issues
            )
            if records_evaluable
            else None
        ),

    "source_location_issue_count":
        (
            len(
                source_location_issues
            )
            if records_evaluable
            else None
        ),

    "duplicate_record_key_count":
        (
            len(
                duplicate_record_keys
            )
            if records_evaluable
            else None
        ),

    "daily_threshold_separate_record_detected":
        (
            bool(
                daily_threshold_separate_record_detected
            )
            if records_evaluable
            else None
        ),

    "gross_reserve_unit_valid":
        (
            bool(
                gross_reserve_unit_valid
            )
            if records_evaluable
            else None
        ),

    "chart_caption_record_count_valid":
        (
            bool(
                chart_caption_record_count_valid
            )
            if records_evaluable
            else None
        ),

    "unsupported_chart_estimation_detected":
        (
            bool(
                unsupported_chart_estimation_detected
            )
            if records_evaluable
            else None
        )
}


print(
    "Structural checks:"
)

print(
    json.dumps(
        STRUCTURAL_CHECKS,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "\nContent diagnostics:"
)

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "\nStructurally_evaluable:",
    structurally_evaluable
)

In [ ]:
# ============================================================
# 29. Technical diagnostic summary
# ============================================================

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "valid_json":
        valid_json,

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        top_level_object_valid,

    "document_id_present":
        document_id_present,

    "document_id_correct":
        document_id_correct,

    "branch_present":
        branch_present,

    "branch_correct":
        branch_correct,

    "records_present":
        records_present,

    "records_is_list":
        records_is_list,

    "records_evaluable":
        records_evaluable,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        (
            number_of_records

            if records_evaluable

            else None
        ),

    "record_count_valid":
        (
            record_count_valid

            if records_evaluable

            else None
        ),

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        (
            observed_category_counts

            if records_evaluable

            else None
        ),

    "category_counts_valid":
        (
            category_counts_valid

            if records_evaluable

            else None
        ),

    "structural_checks":
        STRUCTURAL_CHECKS,

    "records_with_structure_issues":
        (
            records_with_structure_issues

            if records_evaluable

            else None
        ),

    "record_structure_issues":
        (
            record_structure_issues

            if records_evaluable

            else None
        ),

    "records_with_type_issues":
        (
            records_with_type_issues

            if records_evaluable

            else None
        ),

    "field_type_issue_count":
        (
            len(
                field_type_issues
            )

            if records_evaluable

            else None
        ),

    "field_type_issues":
        (
            field_type_issues

            if records_evaluable

            else None
        ),

    "missing_values_by_field":
        (
            missing_values_by_field

            if records_evaluable

            else None
        ),

    "missing_mandatory_value_count":
        (
            missing_mandatory_value_count

            if records_evaluable

            else None
        ),

    "duplicate_record_key_count":
        (
            len(
                duplicate_record_keys
            )

            if records_evaluable

            else None
        ),

    "duplicate_record_keys":
        (
            duplicate_record_keys

            if records_evaluable

            else None
        ),

    "table_record_count":
        (
            len(
                table_records
            )

            if records_evaluable

            else None
        ),

    "table_structurally_evaluable":
        (
            table_structurally_evaluable

            if records_evaluable

            else None
        ),

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "unexpected_qualifier_count":
        (
            len(
                qualifier_issues
            )

            if records_evaluable

            else None
        ),

    "qualifier_issues":
        (
            qualifier_issues

            if records_evaluable

            else None
        ),

    "source_location_issue_count":
        (
            len(
                source_location_issues
            )

            if records_evaluable

            else None
        ),

    "source_location_issues":
        (
            source_location_issues

            if records_evaluable

            else None
        ),

    "daily_threshold_separate_record_detected":
        (
            bool(
                daily_threshold_separate_record_detected
            )

            if records_evaluable

            else None
        ),

    "gross_reserve_unit_valid":
        (
            bool(
                gross_reserve_unit_valid
            )

            if records_evaluable

            else None
        ),

    "chart_caption_record_count_valid":
        (
            bool(
                chart_caption_record_count_valid
            )

            if records_evaluable

            else None
        ),

    "unsupported_chart_estimation_detected":
        (
            bool(
                unsupported_chart_estimation_detected
            )

            if records_evaluable

            else None
        ),

    "unsupported_chart_records":
        (
            unsupported_chart_records

            if records_evaluable

            else None
        ),

    "structurally_evaluable":
        bool(
            structurally_evaluable
        )
}


TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D5_branch_A_technical_diagnostics.json"
)


with TECHNICAL_DIAGNOSTICS_PATH.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        TECHNICAL_DIAGNOSTICS,
        file,
        ensure_ascii=False,
        indent=2
    )


print(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 30. Experiment metadata update
# ============================================================

EXPERIMENT_METADATA.update({
    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if records_evaluable
            else None
        ),

    "parsed_extraction_sha256":
        (
            PARSED_EXTRACTION_SHA256
            if records_evaluable
            else None
        ),

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        (
            number_of_records
            if records_evaluable
            else None
        ),

    "observed_category_counts":
        (
            observed_category_counts
            if records_evaluable
            else None
        ),

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        )
})


with EXPERIMENT_METADATA_PATH.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        EXPERIMENT_METADATA,
        file,
        ensure_ascii=False,
        indent=2
    )


print(
    "Experiment metadata updated:",
    EXPERIMENT_METADATA_PATH
)

In [ ]:
# ============================================================
# 31. Experiment summary
# ============================================================

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        bool(
            input_integrity_passed
        ),

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        True,

    "diagnostic_text_extraction_applied":
        True,

    "text_extraction_used_as_model_input":
        False,

    "pdf_to_text_conversion_applied":
        False,

    "ocr_applied":
        False,

    "page_cropping_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "json_valid":
        bool(
            valid_json
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        (
            number_of_records

            if records_evaluable

            else None
        ),

    "record_count_matches":
        (
            bool(
                record_count_valid
            )

            if records_evaluable

            else None
        ),

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        (
            observed_category_counts

            if records_evaluable

            else None
        ),

    "category_counts_match":
        (
            bool(
                category_counts_valid
            )

            if records_evaluable

            else None
        ),

    "table_structure_matches_reference":
        (
            bool(
                table_structurally_evaluable
            )

            if records_evaluable

            else None
        ),

    "records_with_structure_issues":
        (
            records_with_structure_issues

            if records_evaluable

            else None
        ),

    "records_with_type_issues":
        (
            records_with_type_issues

            if records_evaluable

            else None
        ),

    "missing_mandatory_value_count":
        (
            missing_mandatory_value_count

            if records_evaluable

            else None
        ),

    "unexpected_qualifier_count":
        (
            len(
                qualifier_issues
            )

            if records_evaluable

            else None
        ),

    "source_location_issue_count":
        (
            len(
                source_location_issues
            )

            if records_evaluable

            else None
        ),

    "duplicate_record_key_count":
        (
            len(
                duplicate_record_keys
            )

            if records_evaluable

            else None
        ),

    "daily_threshold_separate_record_detected":
        (
            bool(
                daily_threshold_separate_record_detected
            )

            if records_evaluable

            else None
        ),

    "gross_reserve_unit_valid":
        (
            bool(
                gross_reserve_unit_valid
            )

            if records_evaluable

            else None
        ),

    "unsupported_chart_estimation_detected":
        (
            bool(
                unsupported_chart_estimation_detected
            )

            if records_evaluable

            else None
        ),

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        (
            PARSED_EXTRACTION_PATH.exists()
            if records_evaluable
            else False
        ),

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, D5 Branch A "
        "direct-PDF execution preservation and technical output "
        "checks only. Agreement with the fixed Stage 1 reference "
        "dataset is evaluated separately in Validation A — D5."
    )
}


EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D5_branch_A_experiment_summary.json"
)


with EXPERIMENT_SUMMARY_PATH.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        EXPERIMENT_SUMMARY,
        file,
        ensure_ascii=False,
        indent=2
    )


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 32. Final experiment summary
# ============================================================

observed_records_display = (
    number_of_records

    if records_evaluable

    else "Not evaluable"
)

record_count_display = (
    record_count_valid

    if records_evaluable

    else "Not evaluable"
)

category_count_display = (
    category_counts_valid

    if records_evaluable

    else "Not evaluable"
)

table_structure_display = (
    table_structurally_evaluable

    if records_evaluable

    else "Not evaluable"
)


print(
    "=" * 52
)

print(
    "D5 Branch A experiment completed"
)

print(
    "=" * 52
)

print(
    f"Input integrity passed   : "
    f"{input_integrity_passed}"
)

print(
    f"Valid JSON               : "
    f"{valid_json}"
)

print(
    f"Records evaluable        : "
    f"{records_evaluable}"
)

print(
    f"Expected records         : "
    f"{EXPECTED_RECORD_COUNT}"
)

print(
    f"Observed records         : "
    f"{observed_records_display}"
)

print(
    f"Record count matches     : "
    f"{record_count_display}"
)

print(
    f"Category counts match    : "
    f"{category_count_display}"
)

print(
    f"Table structure matches  : "
    f"{table_structure_display}"
)

print(
    f"Structurally evaluable    : "
    f"{structurally_evaluable}"
)

print()

print(
    "Content validation performed: False"
)

print(
    "Next step: Validation A — D5"
)

In [ ]:
# ============================================================
# 33. Final artefact inventory
# ============================================================

GENERATED_OUTPUTS = [
    INPUT_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_SUMMARY_PATH
]


if (
    records_evaluable
    and PARSED_EXTRACTION_PATH.exists()
):

    GENERATED_OUTPUTS.append(
        PARSED_EXTRACTION_PATH
    )


print(
    "Generated D5 Branch A files:\n"
)


for output_path in GENERATED_OUTPUTS:

    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )

In [ ]:
# ============================================================
# 34. Download experiment artefacts
# ============================================================

for output_path in GENERATED_OUTPUTS:

    if output_path.exists():

        files.download(
            str(
                output_path
            )
        )